# RetroRestore — Proje Sunumu ve Canlı Demo

**Eski fotoğraf ve videoları iyileştirme:** Real-ESRGAN tabanlı süper çözünürlük, gürültü giderme, renk restorasyonu/renklendirme.
Depo: https://github.com/aysecetin0/retro-restore

Bu not defteri projenin **kendi Python kodunu** çalıştırarak şunları canlı gösterir:

| Bölüm | İçerik |
|---|---|
| 0 | Kurulum (kod + önceden eğitilmiş modeller) |
| 1 | **Web sitesi** — son kullanıcının fotoğraf/video yüklediği arayüz |
| 2 | **Veri seti** — klasör yapısı, GT ↔ bozulmuş örnekler, bozulma zincirleri |
| 3 | **Arka plan akışı** — boru hattının her adımı, süresi ve ara görüntüsü |
| 4 | **Real-ESRGAN içeride** — mimari, parametre sayısı, DNI ile gürültü ayarı |
| 5 | **Yöntem karşılaştırma** — Bicubic, Lanczos, Klasik, FSRCNN, EDSR, Real-ESRGAN |
| 6 | **Video boru hattı** — kare çıkarma → işleme → ses senkronlu birleştirme |
| 7 | **Büyük ölçekli test** — DIV2K (100 görüntü × 4 bozulma grubu = 400 test çifti) |

**Başlamadan:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* · ardından *Çalışma zamanı → Tümünü çalıştır*.


## 0 · Kurulum

In [ ]:
import os, sys, subprocess, json, time
if not os.path.exists('/content/retro-restore'):
    !git clone -q https://github.com/aysecetin0/retro-restore.git /content/retro-restore
else:
    !cd /content/retro-restore && git pull -q
%cd /content/retro-restore
!pip -q install flask lpips
!python scripts/download_models.py --all
sys.path.insert(0, '/content/retro-restore')
from retrorestore.core.device import get_device_info
print(json.dumps(get_device_info().as_dict(), indent=1, ensure_ascii=False))


## 1 · Web sitesi (son kullanıcı arayüzü)
Kullanıcı yalnızca dosyasını yükler, birkaç ayar seçer, sonucu önce/sonra kaydırıcısıyla görür ve indirir.
Arka planda Flask REST API + tek işçili iş kuyruğu çalışır (GPU belleği güvenliği).

In [ ]:
subprocess.run('pkill -f run.py', shell=True)
server = subprocess.Popen([sys.executable, 'run.py', '--host', '0.0.0.0', '--port', '8000'],
                          stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(6)
print(open('/content/server.log').read()[-600:])
from google.colab.output import eval_js
from IPython.display import HTML, display
url = eval_js('google.colab.kernel.proxyPort(8000)')
display(HTML(f'<h2>👉 <a href="{url}" target="_blank">RetroRestore web sitesini aç</a></h2><p>{url}</p>'))


## 2 · Veri seti
Önceden eğitilmiş modeller kullandığımız için veri seti **test/doğrulama** içindir. Temiz yüksek çözünürlüklü görüntülerden (ground truth, GT)
**kayıtlı ve tekrar üretilebilir (seed=2026)** bozulmalarla bozuk sürümler üretilir; her çiftin bozulma zinciri `dataset/manifest.json` dosyasındadır.

| Grup | Neyi test eder? |
|---|---|
| `bicubic_x4` | Yalnızca bicubic küçültme — SRCNN/EDSR'ın eğitildiği "ideal" koşul |
| `realworld_x4`, `realworld_x2` | Real-ESRGAN tarzı 2. derece bozulma: bulanıklık → yeniden boyut → gürültü → JPEG (×2 tur) |
| `old_photo` | Sepya, solma, vinyet, gren, çizik, toz, JPEG |
| `grayscale` | Renklendirme testi |


In [ ]:
import pandas as pd, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
DS = Path('dataset')
man = json.loads((DS/'manifest.json').read_text(encoding='utf-8'))
df = pd.DataFrame(man['images'])
print('Kaynak:', man['source'], '· seed:', man['seed'])
display(df.groupby('group').agg(cift_sayisi=('name','count'), ornek_lr=('size_lr','first'), ornek_gt=('size_gt','first')))
!find dataset -maxdepth 2 -type d | sort

def rgb(p): return cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
groups = ['bicubic_x4', 'realworld_x4', 'realworld_x2', 'old_photo', 'grayscale']
fig, ax = plt.subplots(len(groups), 2, figsize=(11, 4.2*len(groups)))
for r, g in enumerate(groups):
    it = df[(df.group == g) & (df.name == 'astronaut')].iloc[0]
    gt, lr = rgb(DS/it.gt), rgb(DS/it.degraded)
    ax[r,0].imshow(cv2.resize(lr, gt.shape[1::-1], interpolation=cv2.INTER_NEAREST)); ax[r,0].set_title(f'{g} — bozulmuş ({it.size_lr})')
    ax[r,1].imshow(gt); ax[r,1].set_title('Ground truth (' + it.size_gt + ')')
    ax[r,0].set_xlabel(' → '.join(it.degradations)[:150], fontsize=7, wrap=True)
    for a in ax[r]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()


## 3 · Arka plan akışı (bir fotoğrafın boru hattından geçişi)
`PipelineTrace` her adımın adını, parametrelerini, süresini ve ara görüntüsünü kaydeder. Aynı kayıt her web işi için `outputs/jobs/<id>/trace.json` dosyasına da yazılır.

In [ ]:
from retrorestore.core.trace import PipelineTrace
from retrorestore.pipeline.image_pipeline import EnhanceOptions, enhance_image
from retrorestore.utils.io import imread
sample = 'dataset/degraded/old_photo/chelsea.png'
tr = PipelineTrace(snapshot_dir=Path('/content/trace_demo'))
out, info = enhance_image(imread(sample), EnhanceOptions(scale=2, scratch_removal=True), tr)
steps = tr.to_dict()['steps']
display(pd.DataFrame([{'adım': i+1, 'aşama': s['stage'], 'başlık': s['title'], 'süre (sn)': s['seconds'],
                       'durum': s['status'], 'parametreler': s['params']} for i, s in enumerate(steps)]))
shots = [s for s in steps if s['snapshot']]
fig, ax = plt.subplots(1, len(shots), figsize=(4*len(shots), 4))
for a, s in zip(ax, shots):
    a.imshow(rgb('/content/trace_demo/' + s['snapshot'])); a.set_title(f"{s['title']}\n{s['seconds']} sn", fontsize=9); a.axis('off')
plt.tight_layout(); plt.show()
print('Toplam:', tr.total_seconds, 'sn · renk kararı:', info['color_decision'], '· SR:', info['sr'])


## 4 · Real-ESRGAN içeride
* **RRDBNet** (x4plus / x2plus): 23 Residual-in-Residual Dense Block, BatchNorm yok — en yüksek kalite.
* **SRVGGNetCompact** (general-x4v3): düz VGG gövdesi + PixelShuffle — ~14 kat daha az parametre, CPU/video için.
* **DNI (Deep Network Interpolation):** `θ = w·θ_denoise + (1−w)·θ_no-denoise` → web sitesindeki *Gürültü temizleme* kaydırıcısı.
* **Döşemeli çıkarım + FP16 + OOM kurtarma:** büyük görüntüler GPU belleğine sığmasa bile işlenir.

In [ ]:
from retrorestore.methods.realesrgan import get_upsampler
for name in ['realesr-general-x4v3', 'RealESRGAN_x4plus', 'RealESRGAN_x2plus']:
    up = get_upsampler(name)
    n = sum(p.numel() for p in up.model.parameters())
    print(f'{name:24s} {type(up.model).__name__:16s} {n/1e6:6.2f} M parametre · ölçek {up.scale}× · cihaz {up.device} · FP16 {up.half}')

# DNI: aynı gürültülü girdide gürültü giderme gücü 0 / 0.5 / 1
gt = imread('dataset/ground_truth/astronaut.png')[100:228, 180:308]
noisy = np.clip(gt + np.random.default_rng(0).normal(0, 18, gt.shape), 0, 255).astype(np.uint8)
fig, ax = plt.subplots(1, 4, figsize=(16, 4.4))
ax[0].imshow(cv2.cvtColor(cv2.resize(noisy, None, fx=4, fy=4, interpolation=cv2.INTER_NEAREST), cv2.COLOR_BGR2RGB)); ax[0].set_title('Gürültülü girdi (σ=18)')
for a, w in zip(ax[1:], [0.0, 0.5, 1.0]):
    o, _ = get_upsampler('realesr-general-x4v3', w).enhance(noisy, 4)
    a.imshow(cv2.cvtColor(o, cv2.COLOR_BGR2RGB)); a.set_title(f'Real-ESRGAN, gürültü gücü w={w}')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()


## 5 · Yöntem karşılaştırma (tek görüntü, ground truth'a karşı)
PSNR/SSIM ↑ (piksel doğruluğu), LPIPS ↓ (algısal benzerlik). GAN tabanlı yöntemler PSNR'da bicubic'ten düşük çıkabilir ama algısal
kalitede (LPIPS ve görsel) öne geçer → **algı–bozulma ödünleşimi** (Blau & Michaeli, 2018).

In [ ]:
from retrorestore.pipeline.benchmark import run_benchmark
for g in ['bicubic_x4', 'realworld_x4']:
    it = df[(df.group == g) & (df.name == 'astronaut')].iloc[0]
    res = run_benchmark(imread(DS/it.degraded)[:, :, :3], Path(f'/content/bench_{g}'), gt=imread(DS/it.gt)[:, :, :3])
    print(f'\n=== {g}: astronaut ({it.size_lr} → {it.size_gt}) ===')
    t = pd.DataFrame([r for r in res['rows'] if r['status'] == 'ok'])[['label', 'family', 'psnr', 'ssim', 'lpips', 'seconds']]
    display(t.sort_values('lpips'))
    plt.figure(figsize=(16, 9)); plt.imshow(rgb(f'/content/bench_{g}/comparison_grid.jpg')); plt.axis('off'); plt.show()


## 6 · Video boru hattı
`ffprobe` → sabit kare hızında PNG karelerine ayırma → her kare görüntü boru hattından → aynı fps ile H.264 + orijinal ses.
Test klibinde her saniyenin başında **beyaz flaş + bip** vardır: çıktıda ikisi hâlâ çakışıyorsa ses–görüntü senkronu korunmuştur.

In [ ]:
from retrorestore.pipeline.video_pipeline import process_video, probe
from base64 import b64encode
src = Path('dataset/videos/degraded/clip_old_film.mp4')
vt = PipelineTrace(snapshot_dir=Path('/content/video_demo'))
vinfo = process_video(src, Path('/content/video_demo'), EnhanceOptions(scale=2), vt,
                      progress=lambda p, m: print(f'\r[{p*100:5.1f}%] {m}', end=''))
print()
display(pd.DataFrame([{'': 'girdi', **{k: vinfo['video'][k] for k in ['width','height','fps','duration','has_audio','video_codec']}},
                      {'': 'çıktı', **{k: vinfo['output'][k] for k in ['width','height','fps','duration','has_audio','video_codec']}}]))
display(pd.DataFrame([{'adım': s['title'], 'süre (sn)': s['seconds'], 'parametreler': s['params']} for s in vt.to_dict()['steps'] if s['stage'] in ('video', 'inference')]))
def vid(p, w=380): return f'<video width="{w}" controls src="data:video/mp4;base64,{b64encode(open(p,"rb").read()).decode()}"></video>'
display(HTML('<div style="display:flex;gap:12px"><div><b>Önce</b><br>' + vid('/content/video_demo/input_preview.mp4') +
             '</div><div><b>Sonra</b><br>' + vid('/content/video_demo/result.mp4') + '</div></div>'))


## 7 · Büyük ölçekli test: DIV2K doğrulama seti
**DIV2K** (NTIRE 2017) süper çözünürlük literatürünün standart test setidir: 100 adet 2K çözünürlüklü, çeşitli sahneler (insan, doğa, mimari, doku).
Her görüntüden 5 bozulmuş sürüm üretilir → **100 × 4 SR grubu = 400 test çifti** × 7 yöntem ≈ 2800 çıkarım. T4 GPU'da yaklaşık 15–25 dakika sürer.
(Daha hızlı deneme için `LIMIT = 20` yapın.)

In [ ]:
LIMIT = 100   # grup başına görüntü sayısı
!python scripts/download_benchmarks.py
!python scripts/make_dataset.py --source benchmarks/DIV2K_valid_HR --prefix div2k_ --append --no-video --max-side 768 --limit 100
!python scripts/evaluate.py --prefix div2k_ --limit $LIMIT --grids 3 --methods bicubic lanczos classical fsrcnn realesr-general-x4v3 RealESRGAN_x4plus RealESRGAN_x2plus


In [ ]:
rep = json.loads(Path('dataset/results/metrics.json').read_text(encoding='utf-8'))
S = pd.DataFrame(rep['summary'])
print(f"{rep['n_unique']} farklı görüntü · {rep['n_images']} test çifti · {rep['device']}")
display(S.pivot_table(index='label', columns='group', values=['psnr', 'ssim', 'lpips']).round(3))
display(S.pivot_table(index='label', columns='group', values='seconds').round(3).rename(columns=lambda c: f'{c} (sn)'))
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
for a, (m, better) in zip(ax, [('psnr', '↑'), ('ssim', '↑'), ('lpips', '↓')]):
    S.pivot_table(index='group', columns='label', values=m).plot.bar(ax=a, rot=0, legend=(m == 'lpips'))
    a.set_title(f'{m.upper()} {better}'); a.set_xlabel('')
ax[2].legend(fontsize=8, loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()
for g in rep['grids']:
    plt.figure(figsize=(16, 9)); plt.imshow(rgb('dataset/results/' + g)); plt.title(g); plt.axis('off'); plt.show()


## 8 · Sonuçları indir (rapor için tablo, CSV ve karşılaştırma görselleri)

In [ ]:
!cd dataset && zip -qr /content/retrorestore_sonuclar.zip results manifest.json
from google.colab import files
files.download('/content/retrorestore_sonuclar.zip')
